In [1]:
!pip install -q langchain-community langchain-text-splitters langchain-huggingface langchain-chroma langchain-groq tavily-python pymupdf python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\shara\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Normal RAG

In [2]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
load_dotenv()
loader = PyMuPDFLoader("paper.pdf")
docs = loader.load()
print(f"Total documents loaded: {len(docs)}")
print("\nFirst 500 chars of page 1:\n", docs[0].page_content[:500])

C:\Users\shara\AppData\Local\Temp\ipykernel_5608\3025071501.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
C:\Users\shara\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total documents loaded: 16

First 500 chars of page 1:
 Corrective Retrieval Augmented Generation
Shi-Qi Yan1*, Jia-Chen Gu2*, Yun Zhu3, Zhen-Hua Ling1
1National Engineering Research Center of Speech and Language Information Processing,
University of Science and Technology of China, Hefei, China
2Department of Computer Science, University of California, Los Angeles
3Google DeepMind
yansiki@mail.ustc.edu.cn, gujc@ucla.edu, yunzhu@google.com, zhling@ustc.edu.cn
Abstract
Large language models (LLMs) inevitably
exhibit hallucinations since the accuracy o


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = splitter.split_documents(docs)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 144


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6387.71it/s]


In [ ]:
test_question = "How does CRAG correct an incorrect retrieval result?"
retrieved_docs = retriever.invoke(test_question)
print(f"Retrieved {len(retrieved_docs)} chunks.")
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:300])

Retrieved 4 chunks.

--- Chunk 1 ---
that the efficacy of CRAG was easily affected by
the accuracy of the retrieval evaluator. The reason
might be the distinct knowledge switch for all input
cases, regardless of the level of confidence in their
judgment. The design of the Ambiguous action

--- Chunk 2 ---
Incorrect
x
kin
+
x
kin
+
Generator
kex
+
x
kex
+
Figure 2: An overview of the proposed CRAG at inference. A retrieval evaluator is constructed to evaluate the
relevance of the retrieved documents to the input, and estimate a confidence degree based on which different
knowledge retrieval actions of 

--- Chunk 3 ---
52.2
53.8
Self-CRAG
49.0
61.8
Self-RAG
29.0
54.9
Self-RAG w. web
24.9
57.9
Table 5: Comparison results between CRAG, Self-
CRAG and RAG, Self-RAG with the same input in
terms of accuracy.
retrieval performance. A part of accurate retrieval
results were deliberately removed at random to
imitate a low

--- Chunk 4 ---
CRAG on the PopQA dataset.
It can be seen
that the genera

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="openai/gpt-oss-safeguard-20b",
    temperature=0
)
prompt = PromptTemplate(
    template="""You are an AI research assistant.
Answer only from the provided context.

Context:
{context}

Question:
{question}

Answer:""",
    input_variables=["context", "question"]
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
parser = StrOutputParser()

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | parser
)
baseline_response = rag_chain.invoke("How does CRAG correct an incorrect retrieval result?")
print("Baseline RAG Response:\n", baseline_response)

Baseline RAG Response:
 CRAG corrects an irrelevant (incorrect) retrieval by first **evaluating** the retrieved document with its *retrieval‑evaluator*.  
The evaluator scores the document’s relevance to the query and produces a confidence estimate.  
If the confidence falls below a threshold, the system triggers the **“Incorrect” action**.  
In this mode the CRAG pipeline does not simply discard the document; instead it feeds the low‑confidence
retrieval into a **correction module** (a T5‑based generator) that rewrites or augments the content so that it
becomes a valid, query‑relevant piece of knowledge.  
Thus, an incorrect retrieval is “corrected” by generating a new, more accurate knowledge snippet that can be
used by the downstream generator to produce the final answer.


## CRAG

In [7]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate

class CRAGEvaluation(BaseModel):
    action: str = Field(description="CORRECT, AMBIGUOUS, or INCORRECT")
    score: float = Field(description="Overall retrieval quality from 0.0 to 1.0")
    reason: str = Field(description="Short explanation")

evaluator = llm.with_structured_output(CRAGEvaluation)

eval_prompt = PromptTemplate(
    template="""You are the retrieval evaluator in a Corrective Retrieval Augmented Generation system.

Question:
{question}

Retrieved documents:
{documents}

Evaluate the retrieved documents together.
- CORRECT: The documents contain enough relevant information to answer.
- AMBIGUOUS: The documents contain some relevant information but are incomplete or uncertain.
- INCORRECT: The documents do not contain useful information for answering.

Judge ONLY the retrieved documents.
""",
    input_variables=["question", "documents"]
)

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
class CRAGEvaluation(BaseModel):
    action: str = Field(description="CORRECT, AMBIGUOUS, or INCORRECT")
    score: float = Field(description="Overall retrieval quality from 0.0 to 1.0")
    reason: str = Field(description="Short explanation")
evaluator = llm.with_structured_output(CRAGEvaluation)
eval_prompt = PromptTemplate(
    template="""You are the retrieval evaluator in a Corrective Retrieval Augmented Generation system.

Question:
{question}

Retrieved documents:
{documents}

Evaluate the retrieved documents together.
- CORRECT: The documents contain enough relevant information to answer.
- AMBIGUOUS: The documents contain some relevant information but are incomplete or uncertain.
- INCORRECT: The documents do not contain useful information for answering.

Judge ONLY the retrieved documents.
""",
    input_variables=["question", "documents"]
)

In [9]:
def evaluate_retrieval(question):
    docs = retriever.invoke(question)
    documents_str = "\n\n".join(
        f"DOCUMENT {i+1}:\n{doc.page_content}"
        for i, doc in enumerate(docs)
    )
    result = (eval_prompt | evaluator).invoke({
        "question": question,
        "documents": documents_str
    })
    return docs, result

def refine_documents(docs, question):
    refined = []
    for doc in docs:
        result = (eval_prompt | evaluator).invoke({
            "question": question,
            "documents": doc.page_content
        })
        if result.score >= 0.5:
            refined.append(doc.page_content)
    return refined

In [ ]:
## Tavily is external web knowledge source used to correct or supplement bad RAG retrieval.
from tavily import TavilyClient
tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
def crag_answer(question):
    # 1. Retrieve & Evaluate
    docs, evaluation = evaluate_retrieval(question)
    action = evaluation.action.strip().upper()  #->CORRECT/AMBIGUOUS/INCORRECT
    # 2. Branch actions
    if "CORRECT" in action and "INCORRECT" not in action:
        refined_docs = refine_documents(docs, question)
        context = "\n\n".join(refined_docs) if refined_docs else format_docs(docs)
    elif "INCORRECT" in action:
        search_results = tavily.search(query=question, max_results=5)
        context = "\n\n".join(r["content"] for r in search_results["results"])
    else:  # AMBIGUOUS
        refined_docs = refine_documents(docs, question)
        search_results = tavily.search(query=question, max_results=3)
        internal_context = "\n\n".join(refined_docs)
        web_context = "\n\n".join(r["content"] for r in search_results["results"])
        context = f"INTERNAL KNOWLEDGE:\n{internal_context}\n\nEXTERNAL KNOWLEDGE:\n{web_context}"
    # 3. Generate Answer
    final_prompt = prompt.invoke({
        "context": context,
        "question": question
    })
    answer = llm.invoke(final_prompt)    
    return answer.content, evaluation

In [ ]:
question = "How does CRAG correct an incorrect retrieval result?"
answer, evaluation = crag_answer(question)
print("ACTION:", evaluation.action)
print("SCORE:", evaluation.score)
print("REASON:", evaluation.reason)
print("\nCRAG ANSWER:\n", answer)

ACTION: AMBIGUOUS
SCORE: 0.6
REASON: The retrieved documents contain partial information about CRAG’s correction mechanism—specifically that a retrieval evaluator estimates confidence and triggers actions such as Correct, Incorrect, or Ambiguous. However, they lack a detailed description of how the system actually corrects an irrelevant retrieval (e.g., re-retrieval, re-ranking, or other methods). Therefore, the set is incomplete for fully answering the question.

CRAG ANSWER:
 CRAG corrects an incorrect retrieval by first **detecting** that the retrieved documents are unreliable.  
A lightweight retrieval‑evaluator scores the set of documents and, if the confidence falls into the “Incorrect” range, CRAG takes the following steps:

1. **Discard the faulty documents** – the original retrieved set is ignored so it cannot bias the LLM.  
2. **Trigger a corrective search** – CRAG initiates a fresh web‑search (or another external knowledge source) for the same query to obtain fresh, more re

## Comparision Example

In [ ]:
from langchain_core.prompts import PromptTemplate
# Target & Ground Truth
comp_question = "In the 2024 CRAG paper, what specific learning rate, optimizer, and base model were used to train the lightweight retrieval evaluator?"
ground_truth = "The retrieval evaluator is initialized from T5-large (Raffel et al., 2020) and fine-tuned with AdamW using a cosine learning rate schedule starting at 2e-5."
# Strict 1-10 Grader (Penalizes Architecture Hallucinations)
score_prompt = PromptTemplate.from_template("""Compare the candidate answer to the ground truth. Score 1 to 10:
- 8 to 9: Accurate on core facts (identifies T5-large and the ~2e-5 schedule).
- 4 to 6: Mostly correct base facts, but misses optimizer or exact schedule.
- 2 to 3: Hallucinates the wrong base model (e.g., MiniLM, BERT), even if other parameters guess lucky matches.
- 1: Completely wrong, irrelevant, or fabricated.

Ground truth: {t}
Answer: {a}

Return ONLY the single integer score:""")

score = lambda a: int(next((w for w in (score_prompt | llm).invoke({"t": ground_truth, "a": a}).content.split() if w.isdigit()), 1))
# Retrieval & Normal RAG
docs = retriever.invoke(comp_question)
normal_ans = llm.invoke(prompt.invoke({"context": format_docs(docs), "question": comp_question})).content

# C-RAG Evaluation & Routing
doc_str = "\n".join(f"DOC {i+1}: {d.page_content}" for i, d in enumerate(docs))
crag_eval = (eval_prompt | evaluator).invoke({"question": comp_question, "documents": doc_str})
act = crag_eval.action.upper()

if "CORRECT" in act and "INCORRECT" not in act:
    ctx = "\n".join(refine_documents(docs, comp_question))
elif "INCORRECT" in act:
    ctx = "\n".join(r["content"] for r in tavily.search(query=comp_question, max_results=5)["results"])
else:  # AMBIGUOUS
    web = "\n".join(r["content"] for r in tavily.search(query=comp_question, max_results=3)["results"])
    ctx = f"INTERNAL:\n{format_docs(docs)}\nEXTERNAL:\n{web}"
crag_ans = llm.invoke(prompt.invoke({"context": ctx, "question": comp_question})).content

# Evaluation & Output
s_norm, s_crag = score(normal_ans), score(crag_ans)
print(f"Evaluator Decision: {crag_eval.action} (Confidence: {crag_eval.score})")
print(f"Normal RAG Score  : {s_norm}/10")
print(f"C-RAG Score       : {s_crag}/10")
print(f"Net Improvement   : +{s_crag - s_norm}\n")
print(f"--- Normal Answer ---\n{normal_ans}\n")
print(f"--- C-RAG Answer ---\n{crag_ans}")

Evaluator Decision: INCORRECT (Confidence: 0.1)
Normal RAG Score  : 1/10
C-RAG Score       : 9/10
Net Improvement   : +8

--- Normal Answer ---
In the CRAG paper the lightweight retrieval evaluator was built on a 2020‑era BERT‑style encoder (the “BERT‑base‑uncased” model).  It was fine‑tuned with the AdamW optimizer at a learning rate of **1 × 10⁻⁵**.

--- C-RAG Answer ---
The lightweight retrieval evaluator in the 2024 CRAG paper was trained as follows:

| Item | Details |
|------|---------|
| **Base model** | T5‑large (the lightweight T5‑large pre‑trained model) |
| **Optimizer** | AdamW |
| **Learning‑rate schedule** | Cosine schedule starting at **2 × 10⁻⁵** and decaying to **10 % of the peak** (i.e., down to 2 × 10⁻⁶) |

Thus, the evaluator was fine‑tuned from a T5‑large backbone using AdamW with an initial learning rate of 2e‑5 under a cosine decay schedule.
